# 03 · Concentrations and manual review

This notebook closes the project with three steps, deliberately without a scoring model:

1. **Profile**: how do addresses with many enterprises differ from ordinary addresses?
2. **Selection**: a small, transparent set of addresses that deviates from the typical concentration profile.
3. **Manual review**: a review sheet for checking 10–15 of these by hand, and a summary of what that check found.

**Why no score.** Section 1 shows that most concentrations look like accountancy firms and domiciliation offices: established legal entities across many sectors. A score that rewards concentration as such would mostly rank those. One explicit rule, tested by hand, says more about the method's usefulness than a weighted score that nobody checks.

**Why no network.** Linking enterprises through shared addresses and contact details (notebook 01) produced one giant component. Section 2 documents this; fixing it is listed as a next step rather than done here.

## 0 · Configuration

In [ ]:
from pathlib import Path
import hashlib

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = PROJECT / "data" / "processed"
REVIEW_DIR = PROJECT / "data" / "review"          # git-ignored: contains real addresses and names
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
FIGURES = PROJECT / "figures"                    # aggregated figures only: these are published
FIGURES.mkdir(exist_ok=True)

# Selection rule (section 3)
MIN_ENTERPRISES = 10      # a concentration
MIN_RECENT_SHARE = 0.5    # at least half started in the last three years
REVIEW_SAMPLE = 15        # how many to check by hand

PUBLIC_MODE = True

def pseudo(value, prefix=""):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    return prefix + hashlib.sha256(str(value).encode()).hexdigest()[:10]

need = ["address_summary_geo", "components", "enterprise", "enterprise_name", "address_study"]
missing = [n for n in need if not (PROCESSED / f"{n}.parquet").exists()]
assert not missing, f"Run notebooks 01 and 02 first (missing: {missing})"

geo = pd.read_parquet(PROCESSED / "address_summary_geo.parquet")
components = pd.read_parquet(PROCESSED / "components.parquet")
f"{len(geo):,} buildings, {int(geo.n_enterprises.sum()):,} enterprise links"

## 1 · Profile by concentration level

For each concentration level: the share of enterprises started in the last three years, the share of sole proprietorships, and the average number of NACE divisions (sectors) per building. Shares are pooled over all enterprises in the level, not averaged per building.

In [ ]:
LEVELS = ["1", "2-4", "5-9", "10-19", "20+"]
geo["level"] = pd.cut(geo.n_enterprises, [0, 1, 4, 9, 19, float("inf")], labels=LEVELS)

profile = (geo.groupby("level", observed=True)
              .agg(buildings=("building_key", "size"),
                   enterprises=("n_enterprises", "sum"),
                   recent=("n_started_last_3y", "sum"),
                   natural=("n_natural_person", "sum"),
                   nace_divisions=("n_nace_divisions", "mean")))
profile["share_recent"] = (profile.recent / profile.enterprises).round(3)
profile["share_sole_proprietor"] = (profile.natural / profile.enterprises).round(3)
profile["nace_divisions"] = profile.nace_divisions.round(1)
profile = profile.drop(columns=["recent", "natural"])
profile

In [ ]:
# Licence art. 2.8: source and update date of the KBO data go with every published figure
with duckdb.connect(str(PROJECT / "data" / "interim" / "kbo.duckdb"), read_only=True) as meta_con:
    meta = dict(meta_con.execute("SELECT Variable, Value FROM raw_meta").fetchall())
SOURCE_NOTE = (f"Bron: KBO Open Data, FOD Economie (extract {meta.get('ExtractNumber', '?')}, "
               f"stand {meta.get('SnapshotDate', '?')}); adressen: BeST Address (BOSA). Bewerking: eigen analyse.")

fig, ax = plt.subplots(figsize=(8, 4.4))
series = [("share_recent", "Gestart in de laatste 3 jaar", "#2a78d6"),
          ("share_sole_proprietor", "Eenmanszaken", "#eb6834")]
x = range(len(profile))
for col, label, color in series:
    y = profile[col].values
    ax.plot(x, y, color=color, linewidth=2, marker="o", markersize=6, label=label)
    ax.annotate(label, (len(profile) - 1, y[-1]), xytext=(8, 0),
                textcoords="offset points", va="center", fontsize=8, color="#444")
ax.set_xticks(list(x), profile.index)
ax.set_xlabel("Aantal ondernemingen op het gebouw", color="#444")
ax.set_ylabel("Aandeel van de ondernemingen", color="#444")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_ylim(0, None)
ax.grid(axis="y", color="#e5e5e5", linewidth=0.8)
ax.set_axisbelow(True)
for s in ["top", "right", "left"]:
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#999")
ax.tick_params(colors="#666", labelsize=8, length=0)
ax.legend(frameon=False, fontsize=8, loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, borderaxespad=0.2)
ax.set_title("Nieuwe ondernemingen en eenmanszaken naar concentratie op het adres",
             loc="left", fontsize=10, color="#222", pad=24)
fig.text(0.01, -0.02, SOURCE_NOTE, fontsize=7, color="#666")
fig.tight_layout()
fig.savefig(FIGURES / "profiel_concentratie.png", dpi=200, bbox_inches="tight")
plt.show()

**Reading.** If concentration addresses were dominated by newly created shell companies, the share of recent starts would rise with concentration. Check whether it does. A falling share of sole proprietorships together with a rising number of sectors points to accountancy firms, domiciliation offices and business centres: addresses that host many unrelated, established companies. That is the base rate any indicator has to beat.

## 2 · Why the network analysis is not used

Notebook 01 linked enterprises through shared addresses (2–50 enterprises) and shared contact details. The component size distribution shows the problem.

In [ ]:
comp_sizes = pd.cut(components.n_enterprises, [1, 2, 5, 20, 100, float("inf")],
                    labels=["2", "3-5", "6-20", "21-100", ">100"])
display(components.groupby(comp_sizes, observed=True)
                  .agg(components=("component_id", "size"), enterprises=("n_enterprises", "sum")))

largest = components.sort_values("n_enterprises", ascending=False).iloc[0]
share = largest.n_enterprises / components.n_enterprises.sum()
print(f"Largest component: {largest.n_enterprises:,} enterprises across {largest.n_addresses:,} addresses "
      f"and {largest.n_shared_contacts:,} shared contact details ({share:.0%} of all linked enterprises).")

A single component this large is not a cluster: medium-sized hubs (an accountant's phone number, a web builder's domain, an address with a few dozen companies) chain unrelated enterprises together. Making the network usable requires a much stricter limit on shared contact details and probably excluding websites. Listed as a next step.

## 3 · Selection: concentrations with many recent starts

One explicit rule: buildings with at least `MIN_ENTERPRISES` enterprises of which at least `MIN_RECENT_SHARE` started in the last three years. This deviates from the typical concentration profile in section 1, and is a pattern that both new business centres and short-lived front companies can produce. The manual check has to tell them apart.

In [ ]:
geo["share_recent"] = geo.n_started_last_3y / geo.n_enterprises
cand = (geo[(geo.n_enterprises >= MIN_ENTERPRISES) & (geo.share_recent >= MIN_RECENT_SHARE)]
          .sort_values(["n_started_last_3y", "n_enterprises"], ascending=False)
          .copy())
print(f"{len(cand)} buildings selected out of {(geo.n_enterprises >= MIN_ENTERPRISES).sum()} "
      f"with {MIN_ENTERPRISES}+ enterprises")

display(cand.groupby("building_size", observed=True).size().rename("buildings").to_frame())
display(cand.groupby(cand.match_stage.fillna("unmatched")).size().rename("buildings").to_frame())

shown = cand[["building_key", "postcode", "n_enterprises", "n_started_last_3y", "share_recent",
              "n_natural_person", "n_nace_divisions", "building_size", "match_stage"]].head(25).copy()
if PUBLIC_MODE:
    shown["building_key"] = shown.building_key.map(lambda v: pseudo(v, "bui_"))
shown.round({"share_recent": 2})

## 4 · Review sheet

The sheet below is for checking addresses by hand. It is written to `data/review/`, which is git-ignored because it contains real addresses and company names. The sample takes the selected buildings with the most recent starts first.

For each address, consult open sources and fill in:

| Column | What to enter |
|---|---|
| `category` | one of: `group_structure` (funds, holdings, project companies of one group), `residential_building` (residential or mixed building, e.g. associations of co-owners), `business_centre` (business centre or domiciliation), `accountancy`, `shared_residence`, `not_pursued` |
| `sources` | what you consulted, e.g. `KBO; Staatsblad; NBB jaarrekening` |
| `consulted_on` | date of consultation |
| `notes` | what you saw, in a sentence or two |

`not_pursued` means: not explained by the register data, and deliberately not investigated further. It is not a conclusion that anything is wrong.

The companion file lists the enterprises at each address (name, start date, legal form, main NACE code) to make the check faster.

In [ ]:
review = cand.head(REVIEW_SAMPLE)[["building_key", "postcode", "municipality", "n_enterprises",
                                   "n_started_last_3y", "n_nace_divisions", "building_size", "lat", "lon"]].copy()
review.insert(0, "review_id", [f"R{i:02d}" for i in range(1, len(review) + 1)])
for col in ["category", "sources", "consulted_on", "notes"]:
    review[col] = ""

sheet_path = REVIEW_DIR / "review_sheet.csv"
assert len(review), "No buildings selected: loosen MIN_ENTERPRISES / MIN_RECENT_SHARE before making a review sheet."
if sheet_path.exists():
    # Never overwrite a sheet that may contain your notes; warn if the selection has changed
    existing = set(pd.read_csv(sheet_path, dtype=str).building_key)
    if existing != set(review.building_key):
        print(f"WARNING: {sheet_path.name} was made for a different selection. "
              "Rename or delete it to make a new one; your filled-in notes are in that file.")
    else:
        print(f"{sheet_path.name} already exists and matches the selection; not overwritten.")
else:
    review.to_csv(sheet_path, index=False)
    print("Written:", sheet_path)

# Companion: enterprises at each review address
addr = pd.read_parquet(PROCESSED / "address_study.parquet", columns=["enterprise_id", "building_key"])
ent = pd.read_parquet(PROCESSED / "enterprise.parquet")
names = pd.read_parquet(PROCESSED / "enterprise_name.parquet")
detail = (review[["review_id", "building_key"]]
          .merge(addr.drop_duplicates(), on="building_key")
          .merge(ent[["enterprise_id", "is_natural_person", "juridical_form_label", "start_date", "nace_code"]],
                 on="enterprise_id", how="left")
          .merge(names, on="enterprise_id", how="left")
          .sort_values(["review_id", "start_date"], ascending=[True, False]))
detail.to_csv(REVIEW_DIR / "review_enterprises.csv", index=False)
print("Written:", REVIEW_DIR / "review_enterprises.csv", f"({len(detail)} rows)")

**Working method for the check.** Keep a source log while you go: for each address, what you looked at, when, and what it showed. Staatsblad publication dates matter: an address can have changed function since the companies registered there. Don't contact businesses and don't visit addresses; this is a desk check of open sources only. Where an address turns out to be a private residence of identifiable people, stop there (see the report).

## 5 · Results of the manual check

Run this after filling in the sheet. It summarises the categories; this table, not any individual case, is the result to publish.

In [ ]:
# Labels as used in the table in RAPPORT.md
CATEGORIES = {
    "group_structure":      "Fondsen, holdings en projectvennootschappen van één groep",
    "residential_building": "Woongebouw of gemengd gebouw (verenigingen van mede-eigenaars)",
    "business_centre":      "Ondernemingscentrum / domiciliëring",
    "accountancy":          "Boekhoudkantoor",
    "shared_residence":     "Waarschijnlijk gedeelde woning",
    "not_pursued":          "Niet verklaard met registergegevens, niet verder onderzocht",
}
filled = pd.read_csv(sheet_path, dtype=str).fillna("")
done = filled[filled.category.str.strip() != ""]
invalid = sorted(set(done.category) - set(CATEGORIES))
if invalid:
    print("Unknown categories (fix spelling):", invalid)

if done.empty:
    print("Review sheet not filled in yet.")
else:
    summary = (done.category.value_counts()
                   .reindex(list(CATEGORIES), fill_value=0)
                   .rename("addresses").to_frame())
    summary.insert(0, "label", summary.index.map(CATEGORIES))
    summary["share"] = (summary.addresses / len(done)).round(2)
    print(f"{len(done)} of {len(filled)} addresses reviewed")
    display(summary)